# EXAMEN - Convocatoria 1 - Programación
Resolver el siguientes problemas de regresión lineal y clasificación.

## 1. Problema de Regresión lineal:

### Imports

In [45]:
import numpy as np

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

### 1.1. Generación de los datos:

Mediante la función `make_regressión` de sklearn, consultar su como emplearla para generar un conjunto de 2000 datos, 10 características, un ruido de 1 y fijar el parámetro ramdom_state=42.

In [23]:
X, y = make_regression(n_samples=2000, n_features=10, noise=1, random_state=42)
print(f"X: {X.shape}, Y: {y.shape}")

X: (2000, 10), Y: (2000,)


### 1.2. Partición de datos externa.
Realizar una partición externa de tipo hold-out seleccionando un 20% de los datos para test (fijar una semilla en 42).

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (1600, 10), X_test: (400, 10)


### 1.3. Estandarización de los datos de train y test.
Utilizar el método StandardScaler().

In [25]:
std_scaling = StandardScaler()

X_train = std_scaling.fit_transform(X_train)
X_test = std_scaling.transform(X_test)

### 1.4. Creación de modelos e hiperparametrización mediante el método de sklearn GridSearchCV:

Instrucciones:

- Cree los siguientes modelos de regresión en un diccionario:
  - Regresión Lineal (OLS).
  - Regresión Lasso
  - Regresión Rígida
  - Vecinos más cercanos (KNN)

- Cree las siguiente métricas en un diccionario para evaluar:
  - MAE
  - MSE
  - RMSE
  - R2

- Haga una búsqueda de los siguientes hiperparámetros mediante GridSearchCV:

```
# Hiperparametros
parameters = {'OLS':{},
              'Lasso':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'Ridge':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'KNN':{'n_neighbors':np.arange(1,15),
                     'weights':('uniform','distance'),
                     'metric':('euclidean','manhattan','cosine')
                     }
              }
```

In [93]:
# Algoritmos
algorithms = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(random_state=42),
    'Ridge': Ridge(random_state=42),
    'KNN': KNeighborsRegressor()
}

# Metricas
metrics = {
  'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
  'MSE': make_scorer(mean_squared_error, greater_is_better=False),
  'RMSE': make_scorer(root_mean_squared_error, greater_is_better=False),
  'R2': make_scorer(r2_score)}

# Hiperparametros
parameters = {'OLS':{},
              'Lasso':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'Ridge':{'alpha':(0.1,0.5,1,5,10,50,100)},
              'KNN':{'n_neighbors':np.arange(1,15),
                     'weights':('uniform','distance'),
                     'metric':('euclidean','manhattan','cosine')
                     }
              }

### 1.5. Evaluación de los modelos sobre el conjunto de test.
- Entrenar los modelos anteriores utilizando todos los datos de entrenamiento y quédese con el mejor modelo de cada uno de los modelos de regresión arrojados por GridSearchCV. Es decir, debe tener un mejor modelo por cada regresión (OLS, Lasso, Rígida y KNN)
- Evaluar su rendimiento sobre el conjunto de test mostrando para cada uno de los modelos

In [94]:
model = {}

for name, _ in algorithms.items():
    print(10*'*', 'Algorithm '+name, 10*'*')
    model_cv = GridSearchCV(algorithms[name], parameters[name], scoring=metrics, refit="R2", cv=None, return_train_score=True)
    model[name] = model_cv.fit(X_train, y_train)
    
    cv_res = model[name].cv_results_
    print("Best parameters:", model[name].best_params_)
    
    best_iter = model[name].best_index_
    for metric in metrics:
        print(f'mean {metric}:', 
              cv_res['mean_test_'+metric][best_iter], 
              f'- std {metric}:', 
              cv_res['std_test_'+metric][best_iter])


********** Algorithm OLS **********
Best parameters: {}
mean MAE: -0.8142624731015278 - std MAE: 0.03162529766844065
mean MSE: -1.0200914370487602 - std MSE: 0.08514697389760788
mean RMSE: -1.0091424509177815 - std RMSE: 0.041508442567908886
mean R2: 0.9999743713478075 - std R2: 3.479122768024652e-06
********** Algorithm Lasso **********
Best parameters: {'alpha': 0.1}
mean MAE: -0.8599471781660867 - std MAE: 0.03771256949527331
mean MSE: -1.1242741234520937 - std MSE: 0.1074436866003939
mean RMSE: -1.059151765920439 - std RMSE: 0.04971579426811077
mean R2: 0.9999717251990656 - std R2: 4.285979656411907e-06
********** Algorithm Ridge **********
Best parameters: {'alpha': 0.1}
mean MAE: -0.8147708614034936 - std MAE: 0.03192453016846064
mean MSE: -1.0203725231097542 - std MSE: 0.08623508867081067
mean RMSE: -1.0092609746911048 - std RMSE: 0.04200961884277579
mean R2: 0.9999743624035592 - std R2: 3.514309506910554e-06
********** Algorithm KNN **********
Best parameters: {'metric': 'cosin

In [99]:
best_model_by_algorithm = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=0.1, random_state=42),
    'Ridge': Ridge(alpha=0.1, random_state=42),
    'KNN': KNeighborsRegressor(metric='cosine', n_neighbors=np.int64(14), weights='distance')
}

for name, model in best_model_by_algorithm.items():
    print(10*'*', 'Best Model by Algorithm '+name, 10*'*')
    model.fit(X_train, y_train)
    for metric_name, metric in metrics.items():
        score = metric(model, X_test, y_test)
        print(f'{metric_name}: {score}')

********** Best Model by Algorithm OLS **********
MAE: -0.7731483950692356
MSE: -0.9534213201938901
RMSE: -0.9764329573472468
R2: 0.9999733598934866
********** Best Model by Algorithm Lasso **********
MAE: -0.8030908362134141
MSE: -1.034921791855248
RMSE: -1.017311059536486
R2: 0.9999710826408178
********** Best Model by Algorithm Ridge **********
MAE: -0.7726548252135538
MSE: -0.9524058849052333
RMSE: -0.9759128469823692
R2: 0.999973388266362
********** Best Model by Algorithm KNN **********
MAE: -50.040132608465626
MSE: -4579.891555976434
RMSE: -67.67489605441912
R2: 0.8720305532435857


### 1.6. Interpretación de resultados.
* Justifica cuál de los modelos utilizarías para ponerlo en producción

## 2. Problema de Clasificación con generación de datos sintéticos y selección de características

En esta actividad se trabajará con generación de datos sintéticos para problemas de clasificación utilizando la función `make_classification` de la librería **scikit-learn**.

El objetivo es evaluar si distintos métodos de selección de características y modelos de clasificación son capaces de identificar las variables realmente relevantes dentro de un conjunto de datos generado artificialmente.

### 2.1 Importación de librerías

Importe las siguientes librerías:

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

from sklearn.preprocessing import StandardScaler

### 2.2. Generación del conjunto de datos

2.2.1. Utilice `make_classification` para generar un dataset de clasificación binaria.

Debe definir explícitamente los siguientes parámetros:

- `n_samples`
- `n_features`
- `n_informative`
- `n_redundant`

El dataset debe tener:

- al menos **500 observaciones**
- al menos **10 características**

El objetivo es que existan variables:

- **informativas**
- **redundantes**
- **no informativas**

2.2.2. Explique brevemente los valores elegidos para cada parámetro.


### 2.3. División de los datos.

Divida el dataset en:

- **conjunto de entrenamiento**
- **conjunto de prueba**

El conjunto de entrenamiento se utilizará para:

- selección de características
- búsqueda de hiperparámetros
- entrenamiento de los modelos

El conjunto de prueba se utilizará **únicamente para evaluar el desempeño final de los modelos**.

### 2.4. Selección de caractarística

Aplique un método de selección de características para identificar cuáles variables parecen ser más relevantes para la predicción.

Puede utilizar el método **SelectKBest** de `sklearn.feature_selection`. Este método evalúa cada característica de forma individual mediante una medida estadística y selecciona las **k variables con mayor puntuación**, es decir, aquellas que tienen mayor relación con la variable objetivo.

Utilice como función de puntuación `f_classif`.

Compare las características seleccionadas con las definidas como **informativas** en `make_classification` y discuta si el método logra identificarlas correctamente.

## 2.5. Escalado de datos

Antes de entrenar los modelos, estandarice las variables utilizando **StandardScaler** de `sklearn.preprocessing`.

La estandarización consiste en transformar las variables para que tengan:

- media igual a 0
- desviación estándar igual a 1

Este paso es importante para modelos sensibles a la escala de los datos, como **regresión logística** y **SVM**. Los árboles de decisiones, son independientes del escalado.

El escalado debe realizarse de la siguiente manera:

1. Ajustar el `StandardScaler` **únicamente con el conjunto de entrenamiento**.
2. Aplicar esa transformación tanto al conjunto de entrenamiento como al conjunto de prueba.


### 2.6. Modelos e hiperparametrización

Trabaje con los siguientes modelos.

**Modelos obligatorios**

- Árbol de decisión (`DecisionTreeClassifier`)
- Regresión logística (`LogisticRegression`)

**Modelo bonus**

- Máquina de soporte vectorial (`SVC`)

Para cada modelo, realice una búsqueda de hiperparámetros utilizando **GridSearchCV** sobre el **conjunto de entrenamiento**.

Utilice las siguientes listas de hiperparámetros:

**Árbol de decisión**

- `max_depth`: [3, 5, 10, None]
- `min_samples_split`: [2, 5, 10]
- `criterion`: ["gini", "entropy"]

**Regresión logística**

- `C`: [0.01, 0.1, 1, 10]
- `solver`: ["lbfgs", "liblinear"]

**SVM (bonus)**

- `C`: [0.1, 1, 10]
- `kernel`: ["linear", "rbf"]
- `gamma`: ["scale", "auto"]

Utilice validación cruzada dentro de `GridSearchCV` para determinar la mejor combinación de hiperparámetros.


### 2.7. Evaluación y análisis

Para cada modelo:

- Del punto anterior, obtenga el modelo con **los mejores hiperparámetros**.
- Evalúe el modelo final en el **conjunto de prueba**.
- Realice este proceso en dos escenarios:
  - utilizando **todas las características**
  - utilizando **solo las características seleccionadas**
- Compare los resultados utilizando métricas de clasificación apropiada y discuta:
  - si la selección de características mejora el desempeño del modelo
  - si reduce la complejidad del modelo
  - si permite identificar correctamente las variables informativas generadas en el dataset
  - qué modelo obtiene mejor desempeño después del ajuste de hiperparámetros
